# ALERTE in a sandbox (ITU AI for Good Lab, Colab, Kaggle)

Runs the ALERTE crisis drill with an open AI model served by Ollama, measures the quality targets, and optionally fine-tunes a model with LoRA.

**Runtime:** choose a GPU runtime (Colab: *Runtime > Change runtime type > T4 GPU*). Everything here is free and open source.

*Fictional exercise. Not a real alert.*

In [ ]:
# 1. Get the code: your team's fork of the ITU reference repository (ALERTE lives in the alerte/ folder)
REPO = "https://github.com/CrashingGuru/ITUAIReadiness.git"  # replace with YOUR fork URL
import os
if not os.path.exists("ITUAIReadiness"):
    !git clone -q {REPO} ITUAIReadiness
%cd ITUAIReadiness/alerte
!pip install -q -r requirements.txt

In [ ]:
# 2. Install and start Ollama (same method as the ITU reference notebook)
!sudo apt-get update -qq && sudo apt-get install -y -qq zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
ollama_proc = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
MODEL = "qwen2.5:7b-instruct"   # T4 GPU friendly; try qwen2.5:14b-instruct on bigger GPUs
!ollama pull {MODEL}
!ollama pull bge-m3

In [ ]:
# 3. Point ALERTE to the model
os.environ.update({
    "ALERTE_BRAIN": "llm",
    "ALERTE_LLM_BASE_URL": "http://localhost:11434/v1",
    "ALERTE_LLM_MODEL": MODEL,
    "ALERTE_EMBED": "ollama",
    "ALERTE_EMBED_MODEL": "bge-m3",
})
!python -m alerte doctor --check-llm

In [ ]:
# 4. Build the knowledge base (ALERTE documents + ITU reference corpus) with multilingual vectors
!python -m alerte ingest --itu yes

In [ ]:
# 5. Run the Cameroon drill and its UAE twin with the AI model
!python -m alerte run cm01 --brain llm
!python -m alerte run ae01 --brain llm
!python -m alerte compare cm01 ae01 --out twin_comparison.md

In [ ]:
# 6. Read the latest report here
from pathlib import Path
from IPython.display import HTML, display
latest = sorted(Path("runs").glob("cm01-*"), key=lambda p: p.stat().st_mtime)[-1]
display(HTML(latest.joinpath("report.html").read_text(encoding="utf-8")))

In [ ]:
# 7. Measure the quality targets (T1 retrieval, T2 quotes, T3 planned gaps, T5 stability, T6 time)
!python -m alerte eval --brain llm --repeats 3

## Optional: fine-tune with LoRA on the sandbox GPU
Teaches a small model to cite exact passages and to say "no legal basis found" instead of inventing.

In [ ]:
!python -m alerte export-sft --out sandbox/alerte_sft.jsonl --runs runs/demo-cm01 runs/demo-ae01
!pip install -q "transformers>=4.44" "peft>=0.11" "accelerate>=0.33" datasets bitsandbytes
!python sandbox/finetune_lora.py --data sandbox/alerte_sft.jsonl --model Qwen/Qwen2.5-7B-Instruct --4bit --epochs 2

## Optional: export the knowledge base for the ITU AI for Good Lab (ChromaDB)

In [ ]:
!pip install -q chromadb
!python -m alerte export-chroma --collection alerte_kb --embed ollama --embed-model nomic-embed-text